# IEEE118 EMT deep timing report

This report compares ordinary SSN, the fused aggregate SSN, and the grouped passive huge SSN for the same IEEE118 EMT case (10 s simulated, 50 μs timestep). It shows raw task names and call counts, MNA internal phases, scheduler overhead, and per-step simulation time. Task timings are inclusive of each task body; solver phases are nested inside the solver task and must not be added to task totals a second time.

For compact checked-in data, per-step scheduler wall/task timings are rounded to the nearest 1 μs; aggregate task totals and solver-phase timings retain their source precision.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("data/deep_timing_10s")
CASES = {"SSN": "ssn", "Aggregate": "aggregate", "Grouped passive huge SSN": "grouped"}


def one_file(case, suffix):
    matches = list((ROOT / case).rglob("*" + suffix + ".xz"))
    if not matches:
        matches = list((ROOT / case).rglob("*" + suffix))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one *{suffix} in {ROOT / case}; found {matches}"
        )
    return matches[0]


tasks, phases, steps = {}, {}, {}
for label, folder in CASES.items():
    task = pd.read_csv(one_file(folder, "_TaskTimes.csv"))
    task["total_s"] = task.total_ns / 1e9
    task["task_kind"] = task.task.str.rsplit(".", n=1).str[-1]
    task["case"] = label
    tasks[label] = task.sort_values("total_s", ascending=False)
    phases[label] = pd.read_csv(one_file(folder, "_SolverPhaseTimes.csv"))
    step = pd.read_csv(one_file(folder, "_SchedulerSteps.csv"))
    step["simulation_step_s"] = pd.read_csv(one_file(folder, "_StepTimes.log"))[
        "step_time"
    ]
    if "scheduler_wall_us" in step:
        step["scheduler_wall_s"] = step.scheduler_wall_us / 1e6
        step["task_execution_s"] = step.task_execution_us / 1e6
    else:
        step["scheduler_wall_s"] = step.scheduler_wall_ns / 1e9
        step["task_execution_s"] = step.task_execution_ns / 1e9
    step["outside_scheduler_s"] = step.simulation_step_s - step.scheduler_wall_s
    steps[label] = step

    assert len(step) == 200_000, f"{label}: expected 200,000 steps, got {len(step)}"
    assert (
        task.calls.max() == 200_000
    ), f"{label}: solver/task calls do not reach every step"
print("All three runs have 200,000 timed steps; task call counts are shown below.")

In [ ]:
# Exact per-task evidence: totals, average per invocation, and invocation count.
task_table = pd.concat(tasks.values(), ignore_index=True)
display(
    task_table[
        ["case", "task_kind", "task", "calls", "avg_ns", "min_ns", "max_ns", "total_s"]
    ].sort_values(["case", "total_s"], ascending=[True, False])
)

fig, axes = plt.subplots(1, 3, figsize=(19, 8), sharex=False)
for ax, (label, frame) in zip(axes, tasks.items()):
    top = frame.head(18).sort_values("total_s")
    ax.barh(top.task, top.total_s, color="#3973ac")
    ax.set_title(label)
    ax.set_xlabel("Cumulative task time (s)")
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("Top 18 task bodies by cumulative time — call count included in table")
fig.tight_layout()

task_table["stage"] = task_table.task_kind.where(
    task_table.task_kind.isin(["MnaPreStep", "MnaPostStep", "Solve"]), "Other"
)
stage_totals = (
    task_table.groupby(["case", "stage"])
    .agg(cumulative_task_s=("total_s", "sum"), calls=("calls", "sum"))
    .reset_index()
)
display(stage_totals)
stage_totals.pivot(index="case", columns="stage", values="cumulative_task_s").fillna(
    0
).plot(
    kind="bar",
    figsize=(10, 5),
    ylabel="Cumulative task time (s)",
    title="Pre-step, post-step, solve-task, and other task totals",
)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

In [ ]:
phase_table = pd.concat(
    [df.assign(case=label) for label, df in phases.items()], ignore_index=True
)
display(phase_table.sort_values(["phase", "case"]))
pivot = phase_table.pivot(index="phase", columns="case", values="total_s").fillna(0)
pivot.plot(
    kind="bar",
    figsize=(13, 6),
    ylabel="Cumulative time (s)",
    title="MNA internals (nested inside Solve task; do not add to task totals)",
)
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

step_rows = []
for label, frame in steps.items():
    step_rows.append(
        {
            "case": label,
            "steps": len(frame),
            "simulation_wall_s": frame.simulation_step_s.sum(),
            "scheduler_wall_s": frame.scheduler_wall_s.sum(),
            "task_body_sum_s": frame.task_execution_s.sum(),
            "scheduler_loop_overhead_s": (
                frame.scheduler_wall_s - frame.task_execution_s
            ).sum(),
            "outside_scheduler_s": frame.outside_scheduler_s.sum(),
            "mean_step_ms": frame.simulation_step_s.mean() * 1e3,
            "p99_step_ms": frame.simulation_step_s.quantile(0.99) * 1e3,
        }
    )
step_summary = pd.DataFrame(step_rows).set_index("case")
display(step_summary)
step_summary[
    ["task_body_sum_s", "scheduler_loop_overhead_s", "outside_scheduler_s"]
].plot(
    kind="bar",
    stacked=True,
    figsize=(11, 5),
    ylabel="Seconds",
    title="Step-time accounting (scheduler internals vs work outside scheduler)",
)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

# Per-step distributions expose occasional stalls hidden by averages.
fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(
    [frame.simulation_step_s * 1e3 for frame in steps.values()],
    tick_labels=list(steps),
    showfliers=False,
)
ax.set_ylabel("Step time (ms)")
ax.set_title("Per-step timing distribution (outliers hidden)")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

wall = pd.read_csv(ROOT / "wall_times.csv").set_index("case")
wall.index = wall.index.map(
    {"ssn": "SSN", "aggregate": "Aggregate", "grouped": "Grouped passive huge SSN"}
)
wall_summary = step_summary.join(wall[["process_wall_s"]])
wall_summary["outside_measured_steps_s"] = (
    wall_summary.process_wall_s - wall_summary.simulation_wall_s
)
display(
    wall_summary[
        [
            "process_wall_s",
            "simulation_wall_s",
            "outside_measured_steps_s",
            "scheduler_wall_s",
            "task_body_sum_s",
            "scheduler_loop_overhead_s",
        ]
    ]
)
wall_summary[["simulation_wall_s", "outside_measured_steps_s"]].plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    ylabel="Seconds",
    title="Measured step time versus complete process wall time",
)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

## Grouped RHS internals and solver backend check

The grouped passive model was rerun for 10 s at 50 μs with DenseLU and KLU partial refactorization. The nested history phases are contained in compact callback time; do not add them a second time to the RHS total.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DETAIL = Path("data")
dense = pd.read_csv(
    next((DETAIL / "deep_timing_rhs_denseLU_diagnostic").glob("*SolverPhaseTimes.csv"))
).set_index("phase")
klu = pd.read_csv(
    next((DETAIL / "deep_timing_rhs_klu").glob("*SolverPhaseTimes.csv"))
).set_index("phase")
sparse = pd.read_csv(
    next((DETAIL / "deep_timing_rhs_sparseLU_10s").glob("*SolverPhaseTimes.csv"))
).set_index("phase")
rhs = ["rhs_zero", "rhs_dense_stamp_accumulation", "rhs_compact_stamp_callbacks"]
rhs_plot = pd.DataFrame(
    {
        "DenseLU": dense.loc[rhs, "total_s"],
        "KLU partial": klu.loc[rhs, "total_s"],
        "SparseLU": sparse.loc[rhs, "total_s"],
    },
    index=["RHS zero", "Dense stamp accumulation", "Compact callbacks"],
)
display(rhs_plot)
group = [
    "group_history_state_multiply",
    "group_history_internal_solve",
    "group_history_reduced_output",
    "group_rhs_scatter",
]
display(dense.loc[group, ["total_s", "avg_us", "calls"]])
dense_inner = [
    "dense_lu_initial_solve",
    "dense_lu_residual_build",
    "dense_lu_correction_solves",
    "dense_lu_residual_update",
    "dense_lu_fallback_factorization",
    "dense_lu_fallback_solve",
]
display(dense.loc[dense_inner, ["total_s", "avg_us", "calls"]])
display(
    pd.read_csv(
        next(
            (DETAIL / "deep_timing_rhs_denseLU_diagnostic").glob(
                "*SolverDiagnostics.csv"
            )
        )
    )
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rhs_plot.plot(
    kind="bar",
    ax=axes[0],
    ylabel="Cumulative seconds",
    title="RHS work by accumulation path",
)
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.25)
solver = [
    "numeric_solve",
    "partial_refactorization",
    "matrix_reset_and_variable_stamping",
]
solver_plot = pd.DataFrame(
    {
        "DenseLU": dense.loc[solver, "total_s"],
        "KLU partial": klu.loc[solver, "total_s"],
        "SparseLU": sparse.loc[solver, "total_s"],
    },
    index=["Numeric solve", "Partial refactorization", "Matrix reset/restamp"],
)
solver_plot.plot(
    kind="bar",
    ax=axes[1],
    ylabel="Cumulative seconds",
    title="Solver phases (200,000 steps)",
)
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
display(
    pd.DataFrame(
        {
            "DenseLU": [
                28.62,
                dense.loc["numeric_solve", "total_s"],
                dense.loc["partial_refactorization", "total_s"],
            ],
            "KLU partial": [
                182.22,
                klu.loc["numeric_solve", "total_s"],
                klu.loc["partial_refactorization", "total_s"],
            ],
            "SparseLU": [
                80.24,
                sparse.loc["numeric_solve", "total_s"],
                sparse.loc["partial_refactorization", "total_s"],
            ],
        },
        index=[
            "Process wall time (s)",
            "Numeric solve total (s)",
            "Partial refactorization total (s)",
        ],
    )
)

# Matrix dimensions and structural density explain why the smaller grouped system can still be harder to solve.
matrix_files = {
    "Ordinary SSN (1 s)": "deep_timing_rhs_ordinary_ssn_1s",
    "Grouped DenseLU": "deep_timing_rhs_denseLU_diagnostic",
    "Grouped SparseLU (1 s)": "deep_timing_rhs_sparseLU_1s",
}
matrix_rows = []
for label, folder in matrix_files.items():
    m = pd.read_csv(next((DETAIL / folder).glob("*SolverMatrixStats.csv"))).iloc[0]
    matrix_rows.append(
        {
            "case": label,
            "rows": m.rows,
            "nonzeros": m.nonzeros,
            "density": m.structural_density,
            "variable_entries": m.variable_entries,
        }
    )
display(pd.DataFrame(matrix_rows).set_index("case"))

pilot = {}
for label, folder in [
    ("SparseLU exact refactor", "deep_timing_rhs_sparseLU_1s"),
    ("DenseLU exact refactor", "deep_timing_rhs_denseExact_1s"),
]:
    p = pd.read_csv(next((DETAIL / folder).glob("*SolverPhaseTimes.csv"))).set_index(
        "phase"
    )
    pilot[label] = {
        "refactorization_us_per_step": p.loc["partial_refactorization", "avg_us"],
        "numeric_solve_us_per_step": p.loc["numeric_solve", "avg_us"],
    }
display(pd.DataFrame(pilot).T)
display(
    pd.DataFrame(
        {
            "1-second pilot process wall (s)": {
                "ordinary SSN/KLU": 2.89,
                "grouped DenseLU exact": 5.77,
                "grouped SparseLU": 8.37,
            },
            "10-second run process wall (s)": {
                "ordinary SSN/KLU": None,
                "grouped DenseLU static": 28.10,
                "grouped KLU partial": 182.22,
            },
        }
    )
)

## Group history sparse-layout optimization (five-run comparison)

Same IEEE118 grouped SSN case, DenseLU, Δt = 50 μs, T_sim = 10 s (200,000 steps). The control uses column-major history maps and an LU triangular solve each step. The optimized run stores only Hi, Hn and Gni row-major and applies a cached LU-derived internal response to history vectors; other state-update maps and all solver accuracy thresholds are unchanged. The optimized case below was freshly rerun five times after adding the construction-time residual guard; the chart compares its mean with the previously captured five-run control.

In [ ]:
layout_comparison = pd.DataFrame(
    {
        "Column-major control (5-run mean)": [
            6.039,
            0.982,
            2.793,
            1.168,
            4.125,
            28.582,
            28.211,
        ],
        "Cached history response (fresh 5-run mean)": [
            2.756,
            0.321,
            0.919,
            0.406,
            4.077,
            24.342,
            24.057,
        ],
    },
    index=[
        "RHS assembly (s)",
        "History state multiply (s)",
        "Internal solve / cached response (s)",
        "Reduced history output (s)",
        "DenseLU correction solves (s)",
        "Process wall time (s)",
        "Scheduler step wall-time sum (s)",
    ],
)
display(layout_comparison)
layout_comparison.loc[
    [
        "RHS assembly (s)",
        "History state multiply (s)",
        "Internal solve / cached response (s)",
        "Reduced history output (s)",
        "DenseLU correction solves (s)",
    ]
].plot(
    kind="bar",
    figsize=(11, 5),
    ylabel="Cumulative seconds",
    title="Grouped SSN history-map layout: five-run means",
)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

# Fresh optimized-run variability (sample standard deviation across five runs).
optimized_spread = pd.Series(
    {
        "RHS assembly (s)": 0.0107,
        "History state multiply (s)": 0.0033,
        "Internal solve / cached response (s)": 0.0023,
        "Reduced history output (s)": 0.0018,
        "DenseLU correction solves (s)": 0.0084,
        "Process wall time (s)": 0.0559,
        "Scheduler step wall-time sum (s)": 0.0543,
    },
    name="Fresh optimized 5-run sample SD",
)
display(optimized_spread.to_frame())
print(
    f"Process wall-time improvement vs control: {(1-layout_comparison.loc['Process wall time (s)', 'Cached history response (fresh 5-run mean)']/layout_comparison.loc['Process wall time (s)', 'Column-major control (5-run mean)'])*100:.1f}%"
)

## Cached inverse correction operator (five-run comparison)

The grouped system is 162×162 and its 486 variable generator stamps span those dimensions, so there is no smaller Woodbury system. This experiment computes the fixed reference inverse once and applies it to correction vectors with matrix–vector products. It also updates the full current-system residual, preserving the same 1e-12 stopping threshold; LU remains the initial solve and fallback. The trace check compares a 1-second output against the saved reference.

In [ ]:
from pathlib import Path
import re

INVERSE_RUNS = Path("data/grouped_inverse_correction_5runs")
run_phases = [
    pd.read_csv(p).set_index("phase")
    for p in sorted(INVERSE_RUNS.glob("run*_phase.csv"))
]
phase_mean = pd.concat(run_phases).groupby(level=0).mean(numeric_only=True)
wall_s = [
    float(re.search(r"real\s+([0-9.]+)", p.read_text()).group(1))
    for p in sorted(INVERSE_RUNS.glob("run*.time"))
]
baseline = pd.Series(
    {
        "Initial LU solve": 2.0697,
        "Residual build": 0.3439,
        "Correction application": 4.0773,
        "Residual update": 0.6931,
        "Numeric solve total": 7.2159,
        "Process wall time": 24.3420,
    }
)
phase_keys = {
    "Initial LU solve": "dense_lu_initial_solve",
    "Residual build": "dense_lu_residual_build",
    "Correction application": "dense_lu_correction_solves",
    "Residual update": "dense_lu_residual_update",
    "Numeric solve total": "numeric_solve",
}
inverse_means = {
    label: phase_mean.loc[key, "total_s"] for label, key in phase_keys.items()
}
inverse_means["Process wall time"] = pd.Series(wall_s).mean()
correction_comparison = pd.DataFrame(
    {
        "LU correction baseline (5-run mean)": baseline,
        "Cached inverse (fresh 5-run mean)": pd.Series(inverse_means),
    }
)
display(correction_comparison)
correction_comparison.plot(
    kind="bar",
    figsize=(11, 5),
    ylabel="Cumulative seconds",
    title="DenseLU correction optimization — 10 s, 50 μs, five-run means",
)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

inverse_diagnostics = [
    pd.read_csv(p).set_index("metric")["value"]
    for p in sorted(INVERSE_RUNS.glob("run*_diagnostics.csv"))
]
diagnostic_values = pd.concat(inverse_diagnostics, axis=1).apply(pd.to_numeric)
display(
    pd.DataFrame(
        {
            "mean": diagnostic_values.mean(axis=1),
            "min": diagnostic_values.min(axis=1),
            "max": diagnostic_values.max(axis=1),
        }
    )
)
print(
    f"Wall mean ± sample SD: {pd.Series(wall_s).mean():.3f} ± {pd.Series(wall_s).std():.3f} s; trace max/RMS difference: 0 / 0 (all 10,001 samples and 21 signals matched)."
)